In [2]:
%pylab inline
import torch 
import torch.nn as nn 
import torch.nn.functional as F 
from tqdm import trange 
from utils import *


Populating the interactive namespace from numpy and matplotlib


In [17]:
# Vanilla RNN Model 

class RNN(nn.Module):
    def __init__(self, in_size, h_size, out_size):
        super(RNN, self).__init__()
        self.h_size = h_size
        self.h2h = nn.Linear(h_size, h_size)
        self.i2h = nn.Linear(in_size, h_size)
        self.out_layer = nn.Linear(h_size, out_size)
        self.softmax = nn.LogSoftmax(dim=1)
    def forward(self, x, h):
        h_next = F.tanh(self.i2h(x) + self.h2h(h))      
        out = self.out_layer(h_next)
        out = self.softmax(out)
        return out, h_next
    
    def init_h(self):
        return torch.zeros(1, self.h_size) 
n_hid = 128 
rnn = RNN(n_chars, n_hid, n_categs)


In [8]:
# Training Function 
opt = torch.optim.Adam(rnn.parameters())
loss_function = nn.NLLLoss()
def train(categ_tensor, line_tensor):
    h = rnn.init_h()

    for i in range(line_tensor.size(0)):
        out, h = rnn(line_tensor[i], h)
    
    loss = loss_function(out, categ_tensor)

    # Update Weights 
    opt.zero_grad()
    loss.backward()
    opt.step()

    return out, loss.item()


In [18]:
# Training Loop 
epochs = 100000
steps = epochs / 10 

for i in trange(epochs+1):
    categ, line, categ_tensor, line_tensor = gen_rand_training_example()
    out, loss = train(categ_tensor, line_tensor)

    if i % steps == 0:
        guess, guess_i = categ_from_out(out)
        correct = "CORRECT" if guess == categ else f"WRONG ({categ})"
        print(f"{loss:.3f} {line} / {guess} {correct}")
    



100%|██████████| 10001/10001 [00:10<00:00, 967.52it/s]2.873 Sayegh / Polish WRONG (Arabic)



In [2]:
def predict(line):
    with torch.no_grad():
        hidden = rnn.init_h()
        line_tensor = line2tensor(line)
        
        for i in range(line_tensor.size()[0]):
            output, hidden = rnn(line_tensor[i], hidden)
        
        pred, _ = categ_from_out(output)
        return pred
inpt = input("Where is this name from?\n")
predict(inpt)

NameError: name 'rnn' is not defined

In [ ]:
# LSTM Model
class LSTM(nn.Module):
    def __init__(self, in_size, h_size, out_size, cell_size):
        super(LSTM, self).__init__()
        self.fg = nn.Linear(in_size, h_size)
        self.in_g = nn.Linear(in_size, h_size)
        self.cell_in = nn.Sequential(nn.Tanh(), nn.Linear(cell_size, cell_size))
        self.out_layer = nn.Linear(h_size, out_size)
    
    def forward(self, x, h, c):
        c = (self.fg(x) *
